# Local Document Q&A Bot (RAG Engine) — Exploration

A quick, hands-on look at the pipeline `app.py` runs, using the SAME `src/`
modules the app uses (not a reimplementation):

1. Extract page-labeled text from a toy in-memory PDF (`src/extraction/pdf_extractor.py`)
2. Chunk it with overlap, preserving page metadata (`src/chunking/chunker.py`)
3. Index the chunks into a temporary ChromaDB collection (`src/vectorstore/store.py`)
4. Run a retrieval query, with and without a document metadata filter
5. Walk through the RAG pipeline's similarity-threshold guardrail
   (`src/rag/pipeline.py`) — the Ollama generation call itself is
   **not** run here (no live Ollama server in this environment), so this
   notebook stops at the retrieval step and inspects the pipeline logic
   directly instead.

This notebook needs internet on its FIRST run only (to download
`paraphrase-MiniLM-L3-v2`, ~61MB) — every run after that is fully offline.
See `src/embeddings/service.py`'s module docstring for why.

In [ ]:
import sys
from pathlib import Path

# Make the project root importable (this notebook lives in notebooks/, one
# level below the project root where config.py and src/ live) so we can
# import the SAME src/ modules app.py uses, rather than duplicating logic.
project_root = Path.cwd().parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

import pandas as pd

from src.chunking.chunker import chunk_page_texts
from src.embeddings.service import embed
from src.vectorstore.store import get_client, list_indexed_sources, query

print("Imports OK — no model downloaded yet (embed() loads it lazily on first call).")

## 1. A Toy "PDF" — Two Short Policy Documents

Real PDF bytes require a file on disk; for a fast, self-contained notebook
demo we skip straight to the shape `src/extraction/pdf_extractor.py`
*produces* — a list of `(page_number, text)` tuples — for two small,
clearly-different documents. This is exactly the input `src/chunking/chunker.py`
expects, so everything downstream (chunking, indexing, retrieval) runs
identically to how it runs on a real uploaded PDF.

In [ ]:
hr_policy_pages = [
    (1, "Employees are entitled to 12 days of paid sick leave per calendar "
        "year. Sick leave must be reported to the manager before 10am on "
        "the day of absence."),
    (2, "Annual leave accrues at 1.5 days per month of service. Unused "
        "annual leave can be carried over, up to a maximum of 5 days, into "
        "the following year."),
]

it_guidelines_pages = [
    (1, "All company laptops must have full-disk encryption enabled before "
        "connecting to the corporate network. IT support is available "
        "Monday through Friday, 9am to 6pm."),
]

print(f"hr_policy.pdf: {len(hr_policy_pages)} page(s)")
print(f"it_guidelines.pdf: {len(it_guidelines_pages)} page(s)")

## 2. Chunk — Paragraph-Aware, With Overlap

`chunk_page_texts()` splits on paragraph breaks first, falling back to
sentence boundaries only when a paragraph itself is too large — never a
naive fixed-size cut mid-sentence (Class 1 Lecture, section 3.4). Every
chunk carries `{source, page}` metadata forward from the input pages.

In [ ]:
hr_chunks = chunk_page_texts(hr_policy_pages, source="hr_policy.pdf")
it_chunks = chunk_page_texts(it_guidelines_pages, source="it_guidelines.pdf")

chunk_table = pd.DataFrame(
    [
        {"source": c.source, "page": c.page, "chars": len(c.text), "text": c.text[:70] + "..."}
        for c in hr_chunks + it_chunks
    ]
)
chunk_table

## 3. Embed + Index Into a Temporary ChromaDB Collection

`index_chunks()` is the ONLY function that calls `model.encode(...)` /
`collection.add(...)` under the hood (see `src/embeddings/service.py` and
`src/vectorstore/store.py`) — this notebook calls it exactly the way
`app.py`'s Upload & Index tab does. We point `config.vectorstore` at a
throwaway temp directory first, so this notebook never touches the app's
real `./data/chroma_db`.

In [ ]:
import tempfile
import config as config_module
from src.vectorstore import store as store_module

# Point the vector store at a temp directory for this notebook run only —
# see src/vectorstore/store.py's lazy-singleton HIGHLIGHTS for why swapping
# get_config() before the first get_client() call is what makes this safe.
_tmp_dir = tempfile.mkdtemp(prefix="rag_notebook_chroma_")
_original_get_config = config_module.get_config
_notebook_config = _original_get_config()
import dataclasses
_notebook_vectorstore_cfg = dataclasses.replace(
    _notebook_config.vectorstore, persist_directory=_tmp_dir, collection_name="notebook_demo"
)
_notebook_config = dataclasses.replace(_notebook_config, vectorstore=_notebook_vectorstore_cfg)
store_module.get_config = lambda: _notebook_config
store_module._client = None
store_module._collection = None

n_indexed = store_module.index_chunks(hr_chunks) + store_module.index_chunks(it_chunks)
print(f"Indexed {n_indexed} chunk(s) into a temp ChromaDB collection at {_tmp_dir}")
print("Indexed sources:", store_module.list_indexed_sources())

## 4. Retrieve — With and Without a Metadata Filter

Ask a question that's really about the HR policy, but restrict the search
to `it_guidelines.pdf` — exactly the "search only this document" feature
from the Ask tab's document picker. The `where={"source": ...}` filter
means the HR chunk is never even considered, no matter how semantically
close it is.

In [ ]:
question = "How many sick leave days do I get?"
question_vector = embed([question])[0]

print("-- Unfiltered (search all documents) --")
for r in store_module.query(question_vector, top_k=2):
    print(f"  {r.source} (page {r.page})  distance={r.distance:.3f}  {r.text[:60]}...")

print("\n-- Filtered to it_guidelines.pdf only --")
for r in store_module.query(question_vector, top_k=2, source_filter="it_guidelines.pdf"):
    print(f"  {r.source} (page {r.page})  distance={r.distance:.3f}  {r.text[:60]}...")

assert store_module.query(question_vector, top_k=2)[0].source == "hr_policy.pdf", (
    "Sanity check: unfiltered retrieval should surface the HR chunk first for this question."
)
print("\nSanity check passed — metadata filtering demonstrably changes retrieval scope.")

## 5. The "Not Found in Documents" Guardrail

`src/rag/pipeline.py`'s `answer_question()` checks the BEST retrieved
chunk's distance against `config.retrieval.max_distance_threshold` *before*
ever calling the LLM. We can observe that check directly here: an
off-topic question against these two small policy documents should come
back with a distance clearly above the threshold, while an on-topic
question stays comfortably below it.

In [ ]:
from config import get_config

cfg = get_config()
threshold = cfg.retrieval.max_distance_threshold

on_topic_vector = embed(["How many sick leave days do I get?"])[0]
off_topic_vector = embed(["What is the capital of France?"])[0]

on_topic_best = store_module.query(on_topic_vector, top_k=1)[0]
off_topic_best = store_module.query(off_topic_vector, top_k=1)[0]

print(f"Guardrail threshold (config.retrieval.max_distance_threshold): {threshold}")
print(f"On-topic best distance:  {on_topic_best.distance:.2f}  -> "
      f"{'ANSWER' if on_topic_best.distance <= threshold else 'NOT FOUND'}")
print(f"Off-topic best distance: {off_topic_best.distance:.2f}  -> "
      f"{'ANSWER' if off_topic_best.distance <= threshold else 'NOT FOUND'}")

assert on_topic_best.distance <= threshold
assert off_topic_best.distance > threshold
print("\nGuardrail behaves as expected on this toy corpus.")

## Takeaways

- `chunk_page_texts()`, `index_chunks()`, and `query()` — the exact
  functions `app.py` uses — chunk, index, and retrieve correctly on a
  small toy corpus, with page metadata preserved end to end.
- The metadata `where` filter demonstrably changes retrieval scope, not
  just display — restricting to `it_guidelines.pdf` returns an IT chunk
  even when an HR chunk would otherwise rank first.
- The similarity-threshold guardrail (`config.retrieval.max_distance_threshold`)
  correctly separates an on-topic question from an off-topic one on this
  corpus — the same check `src/rag/pipeline.py` runs before ever calling
  the LLM.
- Generation itself (`src/llm/client.py`'s `generate()`, calling Ollama) is
  intentionally NOT run in this notebook — this build environment has no
  live Ollama server. `tests/test_pipeline.py` exercises that code path
  fully, with `ollama.chat` mocked.